# KrishiNexus · `krishi-dialect-e5-small-v1`: a dialect-aware retriever on Kaggle

Fine-tunes `intfloat/multilingual-e5-small` to match farmers' questions, in Standard Bangla, Rangpuri, Sylheti,
Bogura (Varendri), Banglish and English, to the app's 18 English knowledge documents. It then exports the encoder to
ONNX with int8 weights and scores that copy through onnxruntime and `tokenizers`, exactly as the app runs it.
TRAINING_PLAN.md section 7b is the design and holds the pre-registered targets.

**Data:** the private Kaggle dataset `krishinexus-dialect-retrieval`: `train.jsonl` and `dev.jsonl` (questions written
by separate writer agents) and `passages.jsonl` (the 18 documents with their context headers).

**Training:** every step scores a batch of questions against all 18 documents (softmax over the whole corpus,
temperature 0.05), so no batch holds a false negative. The epoch is chosen on dev. Dev is also where two baselines are
scored: the zero-shot e5-small and a character n-gram logistic regression trained on the same questions.

**Frozen-test rule:** the test questions are not in this notebook. They are written after this model is frozen and
scored once in the app's full pipeline.

**Before you run:** GPU T4, Internet on, and the dataset attached. Without a GPU the notebook runs a CPU smoke test on
a few rows; its scores are not results.

**Outputs (`/kaggle/working`):** `krishi-dialect-e5-small-v1/` (`model_int8.onnx`, `tokenizer.json`,
`krishi_retriever.json`), `hf-fp32/` (the fine-tuned checkpoint), `eval_report.json`, `dev_predictions.jsonl` and
`run_manifest.json`.


In [ ]:
%pip install -q onnx onnxruntime


In [ ]:
import glob, hashlib, json, math, platform, random, statistics, time
from collections import defaultdict
from pathlib import Path
import numpy as np, torch, transformers
from transformers import AutoModel, AutoTokenizer

GPU = torch.cuda.is_available()
SMOKE = not GPU                     # CPU session: a few rows, one epoch, to exercise every cell; not a result
SEED = 20261113
WORK = Path("/kaggle/working")
ROOT = Path(glob.glob("/kaggle/input/**/passages.jsonl", recursive=True)[0]).parent
BASE = "intfloat/multilingual-e5-small"
OUT = WORK / "krishi-dialect-e5-small-v1"
Q_PREFIX, P_PREFIX, Q_LEN, P_LEN = "query: ", "passage: ", 128, 320
started = time.time()


def read(name):
    return [json.loads(l) for l in (ROOT / name).read_text(encoding="utf-8").splitlines() if l.strip()]


train, dev, passages = read("train.jsonl"), read("dev.jsonl"), read("passages.jsonl")
DOCS = [p["doc_id"] for p in passages]
if SMOKE:
    train, dev = train[:64], dev[:32]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if GPU else "cpu"
print(f"train {len(train)}  dev {len(dev)}  docs {len(DOCS)}  gpu {torch.cuda.get_device_name(0) if GPU else None}")
print(f"torch {torch.__version__}  transformers {transformers.__version__}")


In [ ]:
def metrics(rows, scores):
    # scores: (questions, docs). Rank of the gold doc per question, then top-1 / top-3 / MRR overall and per group.
    gold = np.array([DOCS.index(r["gold_doc"]) for r in rows])
    order = np.argsort(-scores, axis=1, kind="stable")
    rank = np.array([int(np.where(order[i] == gold[i])[0][0]) + 1 for i in range(len(rows))])

    def agg(ix):
        r = rank[np.array(ix, dtype=int)]
        return {"n": int(len(r)), "top1": round(float(np.mean(r == 1)), 4), "top3": round(float(np.mean(r <= 3)), 4),
                "mrr": round(float(np.mean(1 / r)), 4)}

    out = {"all": agg(range(len(rows)))}
    for key in ("dialect", "kind"):
        groups = defaultdict(list)
        for i, row in enumerate(rows):
            groups[row[key]].append(i)
        out[f"by_{key}"] = {g: agg(ix) for g, ix in sorted(groups.items())}
    return out, rank


# Self-check: a perfect scorer gets 1.0, a reversed one gets the gold doc last.
_rows = [{"gold_doc": DOCS[i % len(DOCS)], "dialect": "x", "kind": "y"} for i in range(5)]
_perfect = np.eye(len(DOCS))[[i % len(DOCS) for i in range(5)]]
assert metrics(_rows, _perfect)[0]["all"]["top1"] == 1.0
assert (metrics(_rows, -_perfect)[1] == len(DOCS)).all()


In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
print("longest passage, tokens:", max(len(tok(P_PREFIX + p["text"])["input_ids"]) for p in passages), "(cap", P_LEN, ")")


def load_model():
    return AutoModel.from_pretrained(BASE).to(device)


def encode(model, texts, prefix, max_len, grad=False, bs=64):
    outs = []
    for i in range(0, len(texts), bs):
        batch = tok([prefix + t for t in texts[i:i + bs]], padding=True, truncation=True, max_length=max_len,
                    return_tensors="pt").to(device)
        with torch.set_grad_enabled(grad), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=GPU):
            h = model(**batch).last_hidden_state
        m = batch["attention_mask"].unsqueeze(-1).to(h.dtype)
        e = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)          # mean pooling, as e5 was trained
        outs.append(torch.nn.functional.normalize(e.float(), dim=-1))
    return torch.cat(outs)


def score(model, rows):
    model.eval()
    with torch.no_grad():
        q = encode(model, [r["question"] for r in rows], Q_PREFIX, Q_LEN)
        p = encode(model, [p["text"] for p in passages], P_PREFIX, P_LEN)
    return (q @ p.T).cpu().numpy()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

report = {"model": "krishi-dialect-e5-small-v1", "base_model": BASE, "seed": SEED, "smoke": SMOKE,
          "rows": {"train": len(train), "dev": len(dev)}, "docs": DOCS}

base = load_model()
zero_shot = score(base, dev)
report["dev_zero_shot"], _ = metrics(dev, zero_shot)
del base
if GPU:
    torch.cuda.empty_cache()

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)
clf = LogisticRegression(max_iter=3000, C=10.0).fit(vec.fit_transform([r["question"] for r in train]),
                                                    [r["gold_doc"] for r in train])
proba = clf.predict_proba(vec.transform([r["question"] for r in dev]))
char_lr = np.full((len(dev), len(DOCS)), -1.0)
for j, d in enumerate(clf.classes_):
    char_lr[:, DOCS.index(d)] = proba[:, j]
report["dev_char_ngram_logreg"], _ = metrics(dev, char_lr)
print("zero-shot e5-small:", report["dev_zero_shot"]["all"])
print("char n-gram logreg:", report["dev_char_ngram_logreg"]["all"])


In [ ]:
EPOCHS = 1 if SMOKE else 8
BATCH, LR, TAU = 32, 3e-5, 0.05
model = load_model()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(train) / BATCH)
sched = transformers.get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler("cuda", enabled=GPU)
gold_train = torch.tensor([DOCS.index(r["gold_doc"]) for r in train])
log, best = [], (-1.0, None, 0)
for epoch in range(1, EPOCHS + 1):
    model.train()
    order, losses, t0 = torch.randperm(len(train)), [], time.time()
    for i in range(0, len(train), BATCH):
        ix = order[i:i + BATCH]
        q = encode(model, [train[j]["question"] for j in ix], Q_PREFIX, Q_LEN, grad=True, bs=BATCH)
        p = encode(model, [p["text"] for p in passages], P_PREFIX, P_LEN, grad=True, bs=len(passages))
        loss = torch.nn.functional.cross_entropy(q @ p.T / TAU, gold_train[ix].to(device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        losses.append(loss.item())
    m, _ = metrics(dev, score(model, dev))
    log.append({"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4), "dev_top1": m["all"]["top1"],
                "dev_top3": m["all"]["top3"], "dev_mrr": m["all"]["mrr"], "seconds": round(time.time() - t0, 1)})
    print(log[-1])
    key = m["all"]["top1"] + 0.001 * m["all"]["mrr"]
    if key > best[0]:
        best = (key, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, epoch)
model.load_state_dict(best[1])
report["best_epoch"], report["training_log"] = best[2], log
fine_tuned = score(model, dev)
report["dev_fine_tuned"], dev_rank = metrics(dev, fine_tuned)
print("fine-tuned, epoch", best[2], report["dev_fine_tuned"]["all"])


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(WORK / "hf-fp32")
tok.save_pretrained(WORK / "hf-fp32")
tok.save_pretrained(OUT)                                     # writes tokenizer.json, read by the app with `tokenizers`


class Wrap(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state


cpu_model = model.float().cpu().eval()
dummy = tok(["query: test"], return_tensors="pt")
fp32 = WORK / "model_fp32.onnx"
kw = dict(input_names=["input_ids", "attention_mask"], output_names=["last_hidden_state"], opset_version=17,
          dynamic_axes={n: {0: "batch", 1: "seq"} for n in ("input_ids", "attention_mask", "last_hidden_state")})
args = (dummy["input_ids"], dummy["attention_mask"])
try:
    torch.onnx.export(Wrap(cpu_model), args, str(fp32), dynamo=False, **kw)
except TypeError:                                            # older torch has no `dynamo` argument
    torch.onnx.export(Wrap(cpu_model), args, str(fp32), **kw)

from onnxruntime.quantization import QuantType, quantize_dynamic
quantize_dynamic(str(fp32), str(OUT / "model_int8.onnx"), weight_type=QuantType.QInt8, per_channel=True)  # run 1 per-tensor: 0.875 top-1 agreement
fp32.unlink()
(OUT / "krishi_retriever.json").write_text(json.dumps({
    "base_model": BASE, "query_prefix": Q_PREFIX, "passage_prefix": P_PREFIX, "query_max_len": Q_LEN,
    "passage_max_len": P_LEN, "pooling": "mean", "normalize": True, "trained_docs": DOCS}, indent=2), encoding="utf-8")
print({p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in OUT.iterdir()})


In [ ]:
import onnxruntime as ort
from tokenizers import Tokenizer

sess = ort.InferenceSession(str(OUT / "model_int8.onnx"), providers=["CPUExecutionProvider"])
tk = Tokenizer.from_file(str(OUT / "tokenizer.json"))
tk.enable_padding(pad_id=tk.token_to_id("<pad>"), pad_token="<pad>")


def ort_encode(texts, prefix, max_len):
    # The app's path: `tokenizers` + onnxruntime + mean pooling in numpy.
    tk.enable_truncation(max_len)
    enc = tk.encode_batch([prefix + t for t in texts])
    ids = np.array([e.ids for e in enc], dtype=np.int64)
    mask = np.array([e.attention_mask for e in enc], dtype=np.int64)
    h = sess.run(None, {"input_ids": ids, "attention_mask": mask})[0]
    e = (h * mask[..., None]).sum(1) / np.maximum(mask.sum(1, keepdims=True), 1e-9)
    return e / np.linalg.norm(e, axis=1, keepdims=True)


pv = ort_encode([p["text"] for p in passages], P_PREFIX, P_LEN)
int8 = ort_encode([r["question"] for r in dev], Q_PREFIX, Q_LEN) @ pv.T
report["dev_onnx_int8"], _ = metrics(dev, int8)
report["onnx_int8_top1_agreement_with_checkpoint"] = round(float(np.mean(int8.argmax(1) == fine_tuned.argmax(1))), 4)
single = []
for r in dev[:30]:
    t0 = time.perf_counter()
    ort_encode([r["question"]], Q_PREFIX, Q_LEN)
    single.append((time.perf_counter() - t0) * 1000)
report["onnx_int8_ms_per_question_kaggle_cpu_median"] = round(statistics.median(single), 2)
print("int8 ONNX:", report["dev_onnx_int8"]["all"], "agreement", report["onnx_int8_top1_agreement_with_checkpoint"],
      "ms/question", report["onnx_int8_ms_per_question_kaggle_cpu_median"])


In [ ]:
top = np.argsort(-fine_tuned, axis=1)[:, :3]
with open(WORK / "dev_predictions.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(dev):
        f.write(json.dumps({**r, "rank": int(dev_rank[i]), "top3": [DOCS[j] for j in top[i]],
                            "zero_shot_top1": DOCS[int(zero_shot[i].argmax())],
                            "char_lr_top1": DOCS[int(char_lr[i].argmax())],
                            "int8_top1": DOCS[int(int8[i].argmax())]}, ensure_ascii=False) + "\n")


def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()


(WORK / "eval_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
(WORK / "run_manifest.json").write_text(json.dumps({
    "kernel": "krishinexus-dialect-retriever-training", "smoke": SMOKE, "seed": SEED, "minutes": round((time.time() - started) / 60, 1),
    "gpu": torch.cuda.get_device_name(0) if GPU else None, "python": platform.python_version(),
    "torch": torch.__version__, "transformers": transformers.__version__, "onnxruntime": ort.__version__,
    "files": {p.name: {"bytes": p.stat().st_size, "sha256": sha(p)} for p in sorted(OUT.iterdir())},
}, indent=2), encoding="utf-8")
print(json.dumps({k: report[k]["all"] for k in ("dev_zero_shot", "dev_char_ngram_logreg", "dev_fine_tuned", "dev_onnx_int8")}, indent=2))
